In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                             classification_report, f1_score)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')
matplotlib.use('Agg')

print("Imports good — starting Project 2: Handover Failure Classifier")
print("HO trigger: A3 and A5")
print("TTT values: 160ms, 240ms, 320ms")
print("A3 offset: 3dB")
print("PRB rejection threshold: 60%")

Imports good — starting Project 2: Handover Failure Classifier
HO trigger: A3 and A5
TTT values: 160ms, 240ms, 320ms
A3 offset: 3dB
PRB rejection threshold: 60%


In [2]:
np.random.seed(42)
n_samples = 2000

print("=" * 55)
print("Model A — A3 intra-frequency HO classifier")
print("=" * 55)

# ── A3 MODEL ──────────────────────────────────────────────

def generate_a3_success(n):
    ue_speed = np.random.uniform(0, 120, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))
    ho_config = np.random.choice([0, 1], n, p=[0.5, 0.5])

    serving_rsrp = np.random.normal(-85, 8, n)
    rsrp_delta   = np.random.normal(8, 2, n)        # well above 3dB
    target_rsrp  = serving_rsrp + rsrp_delta

    serving_rsrq = np.random.normal(-9, 1.5, n)     # above -12dB
    rsrq_delta   = np.random.normal(2, 1, n)
    target_rsrq  = serving_rsrq + rsrq_delta

    return pd.DataFrame({
        'serving_rsrp':      serving_rsrp,
        'target_rsrp':       target_rsrp,
        'rsrp_delta':        rsrp_delta,
        'serving_rsrq':      serving_rsrq,
        'target_rsrq':       target_rsrq,
        'rsrq_delta':        rsrq_delta,
        'serving_prb':       np.random.normal(35, 12, n),
        'target_prb':        np.random.normal(30, 10, n),
        'ue_speed':          ue_speed,
        'ttt':               ttt,
        'ho_config':         ho_config,
        'ho_prep_time':      np.random.normal(50, 10, n),
        'time_in_connected': np.random.normal(45, 15, n),
        'label': 'success'
    })

def generate_a3_coverage_gap(n):
    ue_speed = np.random.uniform(0, 150, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))
    ho_config = np.random.choice([0, 1], n, p=[0.5, 0.5])

    serving_rsrp = np.random.normal(-108, 5, n)     # very weak
    rsrp_delta   = np.random.normal(1.5, 1, n)      # below 3dB
    target_rsrp  = serving_rsrp + rsrp_delta

    serving_rsrq = np.random.normal(-14, 1.5, n)    # below -12dB
    rsrq_delta   = np.random.normal(0.5, 1, n)
    target_rsrq  = serving_rsrq + rsrq_delta

    return pd.DataFrame({
        'serving_rsrp':      serving_rsrp,
        'target_rsrp':       target_rsrp,
        'rsrp_delta':        rsrp_delta,
        'serving_rsrq':      serving_rsrq,
        'target_rsrq':       target_rsrq,
        'rsrq_delta':        rsrq_delta,
        'serving_prb':       np.random.normal(72, 10, n),
        'target_prb':        np.random.normal(40, 15, n),
        'ue_speed':          ue_speed,
        'ttt':               ttt,
        'ho_config':         ho_config,
        'ho_prep_time':      np.random.normal(95, 20, n),
        'time_in_connected': np.random.normal(12, 6, n),
        'label': 'coverage_gap'
    })

def generate_a3_too_late(n):
    ue_speed = np.random.uniform(80, 160, n)         # fast UEs
    ttt = np.random.choice([240, 320], n, p=[0.3, 0.7])  # wrong TTT
    ho_config = np.random.choice([0, 1], n, p=[0.5, 0.5])

    serving_rsrp = np.random.normal(-96, 5, n)
    rsrp_delta   = np.random.normal(3.5, 1, n)      # just above 3dB at trigger
    target_rsrp  = serving_rsrp + rsrp_delta

    serving_rsrq = np.random.normal(-11.5, 1, n)    # borderline
    rsrq_delta   = np.random.normal(1, 1, n)
    target_rsrq  = serving_rsrq + rsrq_delta

    return pd.DataFrame({
        'serving_rsrp':      serving_rsrp,
        'target_rsrp':       target_rsrp,
        'rsrp_delta':        rsrp_delta,
        'serving_rsrq':      serving_rsrq,
        'target_rsrq':       target_rsrq,
        'rsrq_delta':        rsrq_delta,
        'serving_prb':       np.random.normal(45, 12, n),
        'target_prb':        np.random.normal(35, 12, n),
        'ue_speed':          ue_speed,
        'ttt':               ttt,
        'ho_config':         ho_config,
        'ho_prep_time':      np.random.normal(130, 25, n),
        'time_in_connected': np.random.normal(7, 3, n),
        'label': 'too_late'
    })

def generate_a3_wrong_target(n):
    ue_speed = np.random.uniform(0, 100, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))
    ho_config = np.zeros(n)                          # RSRP-only — enables failure

    serving_rsrp = np.random.normal(-90, 6, n)
    rsrp_delta   = np.random.normal(5, 1.5, n)      # above 3dB — looks fine
    target_rsrp  = serving_rsrp + rsrp_delta

    serving_rsrq = np.random.normal(-10, 1.5, n)    # serving ok
    target_rsrq  = np.random.normal(-15, 1.5, n)    # target heavily interfered
    rsrq_delta   = target_rsrq - serving_rsrq

    return pd.DataFrame({
        'serving_rsrp':      serving_rsrp,
        'target_rsrp':       target_rsrp,
        'rsrp_delta':        rsrp_delta,
        'serving_rsrq':      serving_rsrq,
        'target_rsrq':       target_rsrq,
        'rsrq_delta':        rsrq_delta,
        'serving_prb':       np.random.normal(40, 12, n),
        'target_prb':        np.random.normal(35, 10, n),
        'ue_speed':          ue_speed,
        'ttt':               ttt,
        'ho_config':         ho_config,
        'ho_prep_time':      np.random.normal(65, 15, n),
        'time_in_connected': np.random.normal(10, 5, n),
        'label': 'wrong_target'
    })

def generate_a3_congestion(n):
    ue_speed = np.random.uniform(0, 100, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))
    ho_config = np.random.choice([0, 1], n, p=[0.5, 0.5])

    serving_rsrp = np.random.normal(-88, 6, n)
    rsrp_delta   = np.random.normal(6, 2, n)        # good delta — radio fine
    target_rsrp  = serving_rsrp + rsrp_delta

    serving_rsrq = np.random.normal(-9.5, 1.5, n)
    rsrq_delta   = np.random.normal(1.5, 1, n)
    target_rsrq  = serving_rsrq + rsrq_delta

    return pd.DataFrame({
        'serving_rsrp':      serving_rsrp,
        'target_rsrp':       target_rsrp,
        'rsrp_delta':        rsrp_delta,
        'serving_rsrq':      serving_rsrq,
        'target_rsrq':       target_rsrq,
        'rsrq_delta':        rsrq_delta,
        'serving_prb':       np.random.normal(45, 12, n),
        'target_prb':        np.random.normal(78, 8, n),   # high — rejects
        'ue_speed':          ue_speed,
        'ttt':               ttt,
        'ho_config':         ho_config,
        'ho_prep_time':      np.random.normal(210, 40, n),
        'time_in_connected': np.random.normal(30, 10, n),
        'label': 'congestion_block'
    })

# Build A3 dataset
df_a3 = pd.concat([
    generate_a3_success(n_samples),
    generate_a3_coverage_gap(n_samples),
    generate_a3_too_late(n_samples),
    generate_a3_wrong_target(n_samples),
    generate_a3_congestion(n_samples)
], ignore_index=True)

# Clip A3 values
df_a3['serving_rsrp'] = df_a3['serving_rsrp'].clip(-130, -50)
df_a3['target_rsrp']  = df_a3['target_rsrp'].clip(-130, -50)
df_a3['serving_rsrq'] = df_a3['serving_rsrq'].clip(-20, -3)
df_a3['target_rsrq']  = df_a3['target_rsrq'].clip(-20, -3)
df_a3['serving_prb']  = df_a3['serving_prb'].clip(0, 100)
df_a3['target_prb']   = df_a3['target_prb'].clip(0, 100)
df_a3['ue_speed']     = df_a3['ue_speed'].clip(0, 200)
df_a3['ho_prep_time'] = df_a3['ho_prep_time'].clip(10, 500)

print(f"A3 dataset shape: {df_a3.shape}")
print(f"Classes: {df_a3['label'].value_counts().to_dict()}")

# ── A5 MODEL ──────────────────────────────────────────────

print("\n" + "=" * 55)
print("Model B — A5 inter-frequency HO classifier")
print("NR-NR inter-frequency | A5-1: -110dBm | A5-2: -100dBm")
print("=" * 55)

def generate_a5_success(n):
    """
    A5 success — both thresholds met cleanly
    Serving clearly below -110, target clearly above -100
    Clean frequency layer change
    """
    ue_speed = np.random.uniform(0, 120, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))

    serving_rsrp = np.random.normal(-115, 3, n)     # clearly below -110
    target_rsrp  = np.random.normal(-94, 3, n)      # clearly above -100

    serving_rsrq = np.random.normal(-13, 1, n)      # poor — driving A5
    target_rsrq  = np.random.normal(-9, 1.5, n)     # target quality good

    # Margin above/below thresholds
    margin_a51   = -110 - serving_rsrp              # how far below -110
    margin_a52   = target_rsrp - (-100)             # how far above -100

    return pd.DataFrame({
        'serving_rsrp':  serving_rsrp,
        'target_rsrp':   target_rsrp,
        'serving_rsrq':  serving_rsrq,
        'target_rsrq':   target_rsrq,
        'margin_a51':    margin_a51,                # key A5 feature
        'margin_a52':    margin_a52,                # key A5 feature
        'a51_threshold': np.full(n, -110.0),
        'a52_threshold': np.full(n, -100.0),
        'target_prb':    np.random.normal(30, 10, n),
        'ue_speed':      ue_speed,
        'ttt':           ttt,
        'ho_prep_time':  np.random.normal(60, 15, n),
        'freq_gap_mhz':  np.random.normal(900, 100, n),  # NR freq separation
        'label': 'success'
    })

def generate_a5_coverage_gap(n):
    """
    A5 coverage gap — condition 1 met but condition 2 fails
    Serving is weak enough (below -110)
    But no target on inter-frequency layer above -100
    UE trapped — serving failing, no escape route
    """
    ue_speed = np.random.uniform(0, 150, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))

    serving_rsrp = np.random.normal(-118, 4, n)     # well below -110
    target_rsrp  = np.random.normal(-106, 4, n)     # below -100 — fails A5-2

    serving_rsrq = np.random.normal(-15, 1.5, n)
    target_rsrq  = np.random.normal(-14, 1.5, n)    # target also poor

    margin_a51   = -110 - serving_rsrp
    margin_a52   = target_rsrp - (-100)             # negative — condition fails

    return pd.DataFrame({
        'serving_rsrp':  serving_rsrp,
        'target_rsrp':   target_rsrp,
        'serving_rsrq':  serving_rsrq,
        'target_rsrq':   target_rsrq,
        'margin_a51':    margin_a51,
        'margin_a52':    margin_a52,
        'a51_threshold': np.full(n, -110.0),
        'a52_threshold': np.full(n, -100.0),
        'target_prb':    np.random.normal(40, 15, n),
        'ue_speed':      ue_speed,
        'ttt':           ttt,
        'ho_prep_time':  np.random.normal(120, 30, n),
        'freq_gap_mhz':  np.random.normal(900, 100, n),
        'label': 'coverage_gap'
    })

def generate_a5_too_late(n):
    """
    A5 too late — thresholds barely met at trigger
    Fast UE — serving degrades further during TTT
    By execution time serving dropped well below -110
    Target also weakened as UE moved
    """
    ue_speed = np.random.uniform(80, 160, n)         # fast UEs only
    ttt = np.random.choice([240, 320], n, p=[0.3, 0.7])

    serving_rsrp = np.random.normal(-111, 2, n)     # just below -110
    target_rsrp  = np.random.normal(-99, 2, n)      # just above -100

    serving_rsrq = np.random.normal(-12.5, 1, n)    # borderline
    target_rsrq  = np.random.normal(-11.5, 1, n)

    margin_a51   = -110 - serving_rsrp              # small margin
    margin_a52   = target_rsrp - (-100)             # small margin

    return pd.DataFrame({
        'serving_rsrp':  serving_rsrp,
        'target_rsrp':   target_rsrp,
        'serving_rsrq':  serving_rsrq,
        'target_rsrq':   target_rsrq,
        'margin_a51':    margin_a51,
        'margin_a52':    margin_a52,
        'a51_threshold': np.full(n, -110.0),
        'a52_threshold': np.full(n, -100.0),
        'target_prb':    np.random.normal(35, 12, n),
        'ue_speed':      ue_speed,
        'ttt':           ttt,
        'ho_prep_time':  np.random.normal(150, 30, n),
        'freq_gap_mhz':  np.random.normal(900, 100, n),
        'label': 'too_late'
    })

def generate_a5_congestion(n):
    """
    A5 congestion — both thresholds met, radio fine
    Target inter-frequency cell PRB > 60%
    Admission rejected — UE stuck on failing serving cell
    Common in dense urban where low-band layer is saturated
    """
    ue_speed = np.random.uniform(0, 100, n)
    ttt = np.where(ue_speed > 80, 160,
          np.where(ue_speed > 40, 240, 320))

    serving_rsrp = np.random.normal(-114, 3, n)     # below -110
    target_rsrp  = np.random.normal(-95, 3, n)      # above -100

    serving_rsrq = np.random.normal(-13, 1, n)
    target_rsrq  = np.random.normal(-9, 1.5, n)     # target quality fine

    margin_a51   = -110 - serving_rsrp
    margin_a52   = target_rsrp - (-100)

    return pd.DataFrame({
        'serving_rsrp':  serving_rsrp,
        'target_rsrp':   target_rsrp,
        'serving_rsrq':  serving_rsrq,
        'target_rsrq':   target_rsrq,
        'margin_a51':    margin_a51,
        'margin_a52':    margin_a52,
        'a51_threshold': np.full(n, -110.0),
        'a52_threshold': np.full(n, -100.0),
        'target_prb':    np.random.normal(76, 8, n),    # high — rejects
        'ue_speed':      ue_speed,
        'ttt':           ttt,
        'ho_prep_time':  np.random.normal(220, 40, n),
        'freq_gap_mhz':  np.random.normal(900, 100, n),
        'label': 'congestion_block'
    })

# Build A5 dataset
df_a5 = pd.concat([
    generate_a5_success(n_samples),
    generate_a5_coverage_gap(n_samples),
    generate_a5_too_late(n_samples),
    generate_a5_congestion(n_samples)        # 4 classes — no wrong_target
], ignore_index=True)

# Clip A5 values
df_a5['serving_rsrp'] = df_a5['serving_rsrp'].clip(-130, -50)
df_a5['target_rsrp']  = df_a5['target_rsrp'].clip(-130, -50)
df_a5['serving_rsrq'] = df_a5['serving_rsrq'].clip(-20, -3)
df_a5['target_rsrq']  = df_a5['target_rsrq'].clip(-20, -3)
df_a5['target_prb']   = df_a5['target_prb'].clip(0, 100)
df_a5['ue_speed']     = df_a5['ue_speed'].clip(0, 200)
df_a5['ho_prep_time'] = df_a5['ho_prep_time'].clip(10, 500)

print(f"A5 dataset shape: {df_a5.shape}")
print(f"Classes: {df_a5['label'].value_counts().to_dict()}")

print("\n=== Both datasets ready ===")
print(f"A3 model: 5 classes, {len(df_a3)} events")
print(f"A5 model: 4 classes, {len(df_a5)} events")
print("\nKey distinguishing features:")
print("A3 — rsrp_delta, target_rsrq, ue_speed, ttt, ho_config")
print("A5 — margin_a51, margin_a52, target_prb, ue_speed, ttt")

Model A — A3 intra-frequency HO classifier
A3 dataset shape: (10000, 14)
Classes: {'success': 2000, 'coverage_gap': 2000, 'too_late': 2000, 'wrong_target': 2000, 'congestion_block': 2000}

Model B — A5 inter-frequency HO classifier
NR-NR inter-frequency | A5-1: -110dBm | A5-2: -100dBm
A5 dataset shape: (8000, 14)
Classes: {'success': 2000, 'coverage_gap': 2000, 'too_late': 2000, 'congestion_block': 2000}

=== Both datasets ready ===
A3 model: 5 classes, 10000 events
A5 model: 4 classes, 8000 events

Key distinguishing features:
A3 — rsrp_delta, target_rsrq, ue_speed, ttt, ho_config
A5 — margin_a51, margin_a52, target_prb, ue_speed, ttt


In [3]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('HO Dataset Validation — A3 and A5 Feature Distributions',
             fontsize=14, fontweight='bold')

colors_class = {
    'success':          '#2ecc71',
    'coverage_gap':     '#e74c3c',
    'too_late':         '#e67e22',
    'wrong_target':     '#9b59b6',
    'congestion_block': '#3498db'
}

# ── A3 plots ──────────────────────────────────────────────

# A3 Plot 1 — RSRP delta distribution
ax = axes[0, 0]
for label in df_a3['label'].unique():
    subset = df_a3[df_a3['label'] == label]['rsrp_delta']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=3, color='black', linewidth=1.5,
           linestyle='--', label='A3 offset 3dB')
ax.set_title('A3 — RSRP delta', fontweight='bold')
ax.set_xlabel('Target minus serving RSRP (dB)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A3 Plot 2 — Target RSRQ
ax = axes[0, 1]
for label in df_a3['label'].unique():
    subset = df_a3[df_a3['label'] == label]['target_rsrq']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=-12, color='black', linewidth=1.5,
           linestyle='--', label='RSRQ threshold -12dB')
ax.set_title('A3 — Target RSRQ', fontweight='bold')
ax.set_xlabel('Target RSRQ (dB)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A3 Plot 3 — UE speed vs TTT
ax = axes[0, 2]
for label in df_a3['label'].unique():
    subset = df_a3[df_a3['label'] == label]
    ax.scatter(subset['ue_speed'], subset['ttt'],
               alpha=0.1, s=2, color=colors_class[label],
               label=label)
ax.set_title('A3 — UE speed vs TTT', fontweight='bold')
ax.set_xlabel('UE speed (km/h)')
ax.set_ylabel('TTT (ms)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A3 Plot 4 — Target PRB
ax = axes[0, 3]
for label in df_a3['label'].unique():
    subset = df_a3[df_a3['label'] == label]['target_prb']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=60, color='black', linewidth=1.5,
           linestyle='--', label='PRB rejection 60%')
ax.set_title('A3 — Target PRB utilization', fontweight='bold')
ax.set_xlabel('Target PRB (%)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# ── A5 plots ──────────────────────────────────────────────

# A5 Plot 1 — Serving RSRP vs A5-1 threshold
ax = axes[1, 0]
for label in df_a5['label'].unique():
    subset = df_a5[df_a5['label'] == label]['serving_rsrp']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=-110, color='black', linewidth=1.5,
           linestyle='--', label='A5-1: -110 dBm')
ax.set_title('A5 — Serving RSRP vs A5-1', fontweight='bold')
ax.set_xlabel('Serving RSRP (dBm)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A5 Plot 2 — Target RSRP vs A5-2 threshold
ax = axes[1, 1]
for label in df_a5['label'].unique():
    subset = df_a5[df_a5['label'] == label]['target_rsrp']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=-100, color='black', linewidth=1.5,
           linestyle='--', label='A5-2: -100 dBm')
ax.set_title('A5 — Target RSRP vs A5-2', fontweight='bold')
ax.set_xlabel('Target RSRP (dBm)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A5 Plot 3 — Margin A5-1 vs Margin A5-2
ax = axes[1, 2]
for label in df_a5['label'].unique():
    subset = df_a5[df_a5['label'] == label]
    ax.scatter(subset['margin_a51'], subset['margin_a52'],
               alpha=0.1, s=2, color=colors_class[label],
               label=label)
ax.axhline(y=0, color='black', linewidth=1,
           linestyle='--', alpha=0.5)
ax.axvline(x=0, color='black', linewidth=1,
           linestyle='--', alpha=0.5)
ax.set_title('A5 — Margin A5-1 vs Margin A5-2', fontweight='bold')
ax.set_xlabel('Margin below A5-1 (dBm)')
ax.set_ylabel('Margin above A5-2 (dBm)')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# A5 Plot 4 — Target PRB
ax = axes[1, 3]
for label in df_a5['label'].unique():
    subset = df_a5[df_a5['label'] == label]['target_prb']
    ax.hist(subset, bins=40, alpha=0.5,
            color=colors_class[label], label=label)
ax.axvline(x=60, color='black', linewidth=1.5,
           linestyle='--', label='PRB rejection 60%')
ax.set_title('A5 — Target PRB utilization', fontweight='bold')
ax.set_xlabel('Target PRB (%)')
ax.set_ylabel('Count')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ho_dataset_validation.png', dpi=150, bbox_inches='tight')
plt.close()

print("Saved: ho_dataset_validation.png")
print("\nWhat to look for in each plot:")
print("A3 RSRP delta    — coverage_gap should sit left of 3dB line")
print("A3 Target RSRQ   — wrong_target should sit left of -12dB line")
print("A3 Speed vs TTT  — too_late cluster at high speed + high TTT")
print("A3 Target PRB    — congestion_block sitting right of 60% line")
print("A5 Serving RSRP  — all classes below -110 dBm line")
print("A5 Target RSRP   — coverage_gap below -100 dBm line")
print("A5 Margin plot   — coverage_gap in bottom quadrant (neg margin_a52)")
print("A5 Target PRB    — congestion_block sitting right of 60% line")

Saved: ho_dataset_validation.png

What to look for in each plot:
A3 RSRP delta    — coverage_gap should sit left of 3dB line
A3 Target RSRQ   — wrong_target should sit left of -12dB line
A3 Speed vs TTT  — too_late cluster at high speed + high TTT
A3 Target PRB    — congestion_block sitting right of 60% line
A5 Serving RSRP  — all classes below -110 dBm line
A5 Target RSRP   — coverage_gap below -100 dBm line
A5 Margin plot   — coverage_gap in bottom quadrant (neg margin_a52)
A5 Target PRB    — congestion_block sitting right of 60% line


In [6]:
df_a3.head()




,serving_rsrp,target_rsrp,rsrp_delta,serving_rsrq,target_rsrq,rsrq_delta,serving_prb,target_prb,ue_speed,ttt,ho_config,ho_prep_time,time_in_connected,label
0,-74.370873,-66.739165,7.631708,-10.281740,-6.883896,3.397844,42.866652,14.554003,44.944814,240,0.0,39.418490,42.963065,success
1,-82.494524,-76.754882,5.739643,-9.604599,-6.413426,3.191173,56.668027,41.522507,114.085717,160,0.0,53.908699,44.272562,success
2,-89.852027,-81.306006,8.546021,-8.591645,-7.133118,1.458527,50.900056,29.183656,87.839273,160,1.0,41.602026,63.828394,success
3,-81.352766,-77.502047,3.850720,-9.085906,-7.187912,1.897994,23.331980,26.333974,71.839018,240,0.0,44.888948,74.529812,success
4,-88.672722,-80.856636,7.816086,-8.177900,-5.506882,2.671018,11.689153,31.853967,18.722237,320,0.0,13.647998,47.990699,success


In [7]:
df_a5.head()

,serving_rsrp,target_rsrp,serving_rsrq,target_rsrq,margin_a51,margin_a52,a51_threshold,a52_threshold,target_prb,ue_speed,ttt,ho_prep_time,freq_gap_mhz,label
0,-109.990042,-90.192081,-14.132045,-12.022631,-0.009958,9.807919,-110.0,-100.0,20.240598,94.647500,160,53.534975,882.091985,success
1,-113.853637,-96.501869,-14.386227,-8.380405,3.853637,3.498131,-110.0,-100.0,25.429001,108.361117,160,25.775422,761.138478,success
2,-115.861646,-95.385420,-13.839020,-8.289576,5.861646,4.614580,-110.0,-100.0,25.576892,38.068364,320,34.808053,824.488912,success
3,-114.253825,-99.324676,-12.405072,-7.664546,4.253825,0.675324,-110.0,-100.0,28.811721,106.430204,160,33.003405,1060.047841,success
4,-113.676224,-91.524399,-14.042171,-10.402328,3.676224,8.475601,-110.0,-100.0,31.662310,31.960829,320,75.493548,865.256692,success


In [8]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report)
import time

print("=" * 55)
print("Training A3 and A5 HO failure classifiers")
print("=" * 55)

# ── A3 MODEL ──────────────────────────────────────────────

# Step 1 — prepare A3 features and labels
feature_cols_a3 = [
    'serving_rsrp', 'target_rsrp', 'rsrp_delta',
    'serving_rsrq', 'target_rsrq', 'rsrq_delta',
    'serving_prb',  'target_prb',
    'ue_speed',     'ttt',
    'ho_config',    'ho_prep_time',
    'time_in_connected'
]

X_a3 = df_a3[feature_cols_a3]
y_a3 = df_a3['label']

# Step 2 — split A3 data
X_a3_train, X_a3_test, y_a3_train, y_a3_test = train_test_split(
    X_a3, y_a3,
    test_size=0.2,
    random_state=42,
    stratify=y_a3
)

print(f"\nA3 training samples: {X_a3_train.shape[0]}")
print(f"A3 test samples:     {X_a3_test.shape[0]}")

# Step 3 — train A3 Random Forest
print("\nTraining A3 Random Forest...")
start = time.time()
clf_a3 = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
clf_a3.fit(X_a3_train, y_a3_train)
a3_train_time = time.time() - start
print(f"Training completed in {a3_train_time:.2f} seconds")

# Step 4 — evaluate A3
y_a3_pred = clf_a3.predict(X_a3_test)
a3_accuracy = accuracy_score(y_a3_test, y_a3_pred)

print(f"\nA3 Overall accuracy: {a3_accuracy*100:.1f}%")
print(f"\nA3 Per-class accuracy:")
for label in df_a3['label'].unique():
    mask = y_a3_test == label
    acc  = accuracy_score(y_a3_test[mask], y_a3_pred[mask])
    print(f"  {label:<20} {acc*100:.1f}%")

# Step 5 — A3 cross validation
print(f"\nA3 Cross-validation (5-fold):")
cv_scores_a3 = cross_val_score(
    clf_a3, X_a3, y_a3, cv=5, scoring='accuracy', n_jobs=-1
)
print(f"  Scores: {[f'{s:.3f}' for s in cv_scores_a3]}")
print(f"  Mean:   {cv_scores_a3.mean():.3f}")
print(f"  Std:    {cv_scores_a3.std():.3f}")

# Step 6 — A3 feature importance
print(f"\nA3 Feature importance:")
a3_importance = pd.Series(
    clf_a3.feature_importances_,
    index=feature_cols_a3
).sort_values(ascending=False)
for feat, imp in a3_importance.items():
    bar = '█' * int(imp * 100)
    print(f"  {feat:<22} {imp:.3f} {bar}")

# ── A5 MODEL ──────────────────────────────────────────────

print("\n" + "=" * 55)
print("A5 model")
print("=" * 55)

# Step 1 — prepare A5 features and labels
feature_cols_a5 = [
    'serving_rsrp', 'target_rsrp',
    'serving_rsrq', 'target_rsrq',
    'margin_a51',   'margin_a52',
    'a51_threshold','a52_threshold',
    'target_prb',   'ue_speed',
    'ttt',          'ho_prep_time',
    'freq_gap_mhz'
]

X_a5 = df_a5[feature_cols_a5]
y_a5 = df_a5['label']

# Step 2 — split A5 data
X_a5_train, X_a5_test, y_a5_train, y_a5_test = train_test_split(
    X_a5, y_a5,
    test_size=0.2,
    random_state=42,
    stratify=y_a5
)

print(f"\nA5 training samples: {X_a5_train.shape[0]}")
print(f"A5 test samples:     {X_a5_test.shape[0]}")

# Step 3 — train A5 Random Forest
print("\nTraining A5 Random Forest...")
start = time.time()
clf_a5 = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
clf_a5.fit(X_a5_train, y_a5_train)
a5_train_time = time.time() - start
print(f"Training completed in {a5_train_time:.2f} seconds")

# Step 4 — evaluate A5
y_a5_pred = clf_a5.predict(X_a5_test)
a5_accuracy = accuracy_score(y_a5_test, y_a5_pred)

print(f"\nA5 Overall accuracy: {a5_accuracy*100:.1f}%")
print(f"\nA5 Per-class accuracy:")
for label in df_a5['label'].unique():
    mask = y_a5_test == label
    acc  = accuracy_score(y_a5_test[mask], y_a5_pred[mask])
    print(f"  {label:<20} {acc*100:.1f}%")

# Step 5 — A5 cross validation
print(f"\nA5 Cross-validation (5-fold):")
cv_scores_a5 = cross_val_score(
    clf_a5, X_a5, y_a5, cv=5, scoring='accuracy', n_jobs=-1
)
print(f"  Scores: {[f'{s:.3f}' for s in cv_scores_a5]}")
print(f"  Mean:   {cv_scores_a5.mean():.3f}")
print(f"  Std:    {cv_scores_a5.std():.3f}")

# Step 6 — A5 feature importance
print(f"\nA5 Feature importance:")
a5_importance = pd.Series(
    clf_a5.feature_importances_,
    index=feature_cols_a5
).sort_values(ascending=False)
for feat, imp in a5_importance.items():
    bar = '█' * int(imp * 100)
    print(f"  {feat:<22} {imp:.3f} {bar}")

# ── Summary ───────────────────────────────────────────────
print("\n" + "=" * 55)
print("Summary")
print("=" * 55)
print(f"A3 model — {len(df_a3['label'].unique())} classes: "
      f"{a3_accuracy*100:.1f}% accuracy")
print(f"A5 model — {len(df_a5['label'].unique())} classes: "
      f"{a5_accuracy*100:.1f}% accuracy")
print(f"\nTraining gap check:")
y_a3_train_pred = clf_a3.predict(X_a3_train)
y_a5_train_pred = clf_a5.predict(X_a5_train)
print(f"  A3 train: {accuracy_score(y_a3_train, y_a3_train_pred)*100:.1f}%  "
      f"test: {a3_accuracy*100:.1f}%  "
      f"gap: {(accuracy_score(y_a3_train, y_a3_train_pred)-a3_accuracy)*100:.1f}%")
print(f"  A5 train: {accuracy_score(y_a5_train, y_a5_train_pred)*100:.1f}%  "
      f"test: {a5_accuracy*100:.1f}%  "
      f"gap: {(accuracy_score(y_a5_train, y_a5_train_pred)-a5_accuracy)*100:.1f}%")

Training A3 and A5 HO failure classifiers

A3 training samples: 8000
A3 test samples:     2000

Training A3 Random Forest...
Training completed in 0.25 seconds

A3 Overall accuracy: 99.7%

A3 Per-class accuracy:
  success              99.8%
  coverage_gap         99.5%
  too_late             99.2%
  wrong_target         99.8%
  congestion_block     100.0%

A3 Cross-validation (5-fold):
  Scores: ['0.996', '0.999', '0.998', '0.998', '0.999']
  Mean:   0.998
  Std:    0.001

A3 Feature importance:
  target_prb             0.156 ███████████████
  rsrq_delta             0.153 ███████████████
  ho_prep_time           0.148 ██████████████
  time_in_connected      0.100 █████████
  target_rsrp            0.096 █████████
  target_rsrq            0.084 ████████
  rsrp_delta             0.059 █████
  serving_rsrp           0.051 █████
  ue_speed               0.048 ████
  serving_prb            0.047 ████
  serving_rsrq           0.035 ███
  ttt                    0.017 █
  ho_config            

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('HO Failure Classifier — Confusion Matrices',
             fontsize=14, fontweight='bold')

# ── A3 Confusion Matrix ────────────────────────────────────

a3_classes = ['success', 'coverage_gap', 'too_late',
              'wrong_target', 'congestion_block']

cm_a3 = confusion_matrix(y_a3_test, y_a3_pred, labels=a3_classes)

sns.heatmap(cm_a3,
            annot=True,
            fmt='d',
            xticklabels=a3_classes,
            yticklabels=a3_classes,
            cmap='Blues',
            ax=axes[0])
axes[0].set_title('A3 intra-frequency HO classifier',
                  fontweight='bold')
axes[0].set_ylabel('Actual failure type')
axes[0].set_xlabel('Predicted failure type')
axes[0].tick_params(axis='x', rotation=45)

# ── A5 Confusion Matrix ────────────────────────────────────

a5_classes = ['success', 'coverage_gap',
              'too_late', 'congestion_block']

cm_a5 = confusion_matrix(y_a5_test, y_a5_pred, labels=a5_classes)

sns.heatmap(cm_a5,
            annot=True,
            fmt='d',
            xticklabels=a5_classes,
            yticklabels=a5_classes,
            cmap='Greens',
            ax=axes[1])
axes[1].set_title('A5 inter-frequency HO classifier',
                  fontweight='bold')
axes[1].set_ylabel('Actual failure type')
axes[1].set_xlabel('Predicted failure type')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('ho_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: ho_confusion_matrices.png")

# ── Classification Report ──────────────────────────────────

print("\n=== A3 Classification Report ===")
print(classification_report(y_a3_test, y_a3_pred,
                             target_names=a3_classes))

print("\n=== A5 Classification Report ===")
print(classification_report(y_a5_test, y_a5_pred,
                             target_names=a5_classes))

# ── Recommendation Engine ──────────────────────────────────

print("\n" + "=" * 55)
print("Recommendation engine")
print("=" * 55)
print("Given a new HO failure — predict root cause")
print("and generate recommended action")

# Recommendation dictionary — maps predicted class to action
recommendations = {
    'success': {
        'priority': 'none',
        'action':   'No action required — HO completed successfully',
        'check':    'Monitor for trend changes'
    },
    'coverage_gap': {
        'priority': 'high',
        'action':   'Coverage gap detected — check antenna tilt and azimuth',
        'check':    'Verify neighbor relations, check feeder/VSWR, '
                    'consider coverage fill node'
    },
    'too_late': {
        'priority': 'medium',
        'action':   'TTT mismatch with UE speed — reduce TTT to 160ms',
        'check':    'Check UE speed profile on this cell, '
                    'verify highway or transit corridor coverage, '
                    'consider A3 offset reduction'
    },
    'wrong_target': {
        'priority': 'medium',
        'action':   'Pilot pollution — enable RSRQ filter in HO config',
        'check':    'Review neighbor cell list, check for missing NR, '
                    'verify RSRQ threshold set to -12dB, '
                    'consider antenna downtilt on interfering cell'
    },
    'congestion_block': {
        'priority': 'high',
        'action':   'Target cell congested — PRB > 60% blocking admission',
        'check':    'Check target cell load trend, '
                    'consider load balancing parameter update, '
                    'capacity expansion or traffic steering to alternate cell'
    }
}

# Simulate 10 new HO failure events
print("\nSimulating 10 new HO events — predicting root cause:\n")

# Sample 10 random test events
sample_idx = np.random.choice(len(X_a3_test), 10, replace=False)
X_sample   = X_a3_test.iloc[sample_idx]
y_actual   = y_a3_test.iloc[sample_idx]
y_sample_pred = clf_a3.predict(X_sample)
y_proba    = clf_a3.predict_proba(X_sample)

print(f"{'Event':<7} {'Actual':<20} {'Predicted':<20} "
      f"{'Confidence':<12} {'Priority'}")
print("-" * 75)

for i, (actual, predicted, proba) in enumerate(
        zip(y_actual, y_sample_pred, y_proba)):
    confidence = proba.max()
    priority   = recommendations[predicted]['priority']
    match      = "✓" if actual == predicted else "✗"
    print(f"  {i+1:<5} {actual:<20} {predicted:<20} "
          f"{confidence*100:<11.1f}% {priority}  {match}")

# Show full recommendation for first failure event
print("\n" + "=" * 55)
print("Sample recommendation — Event 1:")
print("=" * 55)
first_pred = y_sample_pred[0]
rec = recommendations[first_pred]
print(f"  Predicted root cause: {first_pred}")
print(f"  Priority:             {rec['priority']}")
print(f"  Recommended action:   {rec['action']}")
print(f"  What to check:        {rec['check']}")

# Summary across all test predictions
print("\n" + "=" * 55)
print("Full test set — root cause distribution:")
print("=" * 55)
pred_counts = pd.Series(y_a3_pred).value_counts()
for cause, count in pred_counts.items():
    pct = count / len(y_a3_pred) * 100
    rec = recommendations[cause]
    bar = '█' * int(pct / 2)
    print(f"  {cause:<20} {count:>4} ({pct:>5.1f}%)  "
          f"priority: {rec['priority']:<8} {bar}")

Saved: ho_confusion_matrices.png

=== A3 Classification Report ===
                  precision    recall  f1-score   support

         success       1.00      1.00      1.00       400
    coverage_gap       0.99      0.99      0.99       400
        too_late       1.00      1.00      1.00       400
    wrong_target       1.00      0.99      0.99       400
congestion_block       0.99      1.00      1.00       400

        accuracy                           1.00      2000
       macro avg       1.00      1.00      1.00      2000
    weighted avg       1.00      1.00      1.00      2000


=== A5 Classification Report ===
                  precision    recall  f1-score   support

         success       1.00      1.00      1.00       400
    coverage_gap       0.98      0.99      0.99       400
        too_late       0.99      0.99      0.99       400
congestion_block       1.00      0.99      0.99       400

        accuracy                           0.99      1600
       macro avg       0

In [10]:
print("=" * 60)
print("Daily HO failure analysis report")
print("Simulating one day of network HO failures")
print("=" * 60)

# Step 1 — Simulate a realistic daily failure mix
# In a real network failures are NOT equally distributed
# Coverage gap most common, wrong_target least common
# Based on typical 5G NSA network failure distributions

np.random.seed(99)
daily_failures = 500  # realistic daily HO failure count

# Realistic class distribution from your RF experience
a3_daily_dist = {
    'coverage_gap':     0.35,   # most common — cell edge UEs
    'too_late':         0.25,   # highway corridors
    'congestion_block': 0.20,   # busy hour peaks
    'wrong_target':     0.12,   # pilot pollution areas
    'success':          0.08    # borderline events logged
}

a5_daily_dist = {
    'coverage_gap':     0.40,   # NR layer too weak
    'congestion_block': 0.30,   # low band layer saturated
    'too_late':         0.25,   # fast UEs on IFHO
    'success':          0.05    # borderline logged events
}

# Generate A3 daily events
a3_labels = np.random.choice(
    list(a3_daily_dist.keys()),
    size=daily_failures,
    p=list(a3_daily_dist.values())
)

# Generate A5 daily events
a5_daily_count = 200  # fewer A5 events than A3
a5_labels = np.random.choice(
    list(a5_daily_dist.keys()),
    size=a5_daily_count,
    p=list(a5_daily_dist.values())
)

# Step 2 — Generate realistic feature values for each event
# and run through the trained classifiers

def generate_event_features_a3(label, n=1):
    """Generate realistic A3 features for a given failure type"""
    if label == 'coverage_gap':
        return pd.DataFrame({
            'serving_rsrp':      np.random.normal(-108, 5, n),
            'target_rsrp':       np.random.normal(-109, 5, n),
            'rsrp_delta':        np.random.normal(1.5, 1, n),
            'serving_rsrq':      np.random.normal(-14, 1.5, n),
            'target_rsrq':       np.random.normal(-14.5, 1.5, n),
            'rsrq_delta':        np.random.normal(-0.5, 0.5, n),
            'serving_prb':       np.random.normal(72, 10, n),
            'target_prb':        np.random.normal(40, 15, n),
            'ue_speed':          np.random.normal(30, 15, n),
            'ttt':               np.random.choice([160,240,320], n),
            'ho_config':         np.random.choice([0,1], n),
            'ho_prep_time':      np.random.normal(95, 20, n),
            'time_in_connected': np.random.normal(12, 6, n),
        })
    elif label == 'too_late':
        return pd.DataFrame({
            'serving_rsrp':      np.random.normal(-96, 5, n),
            'target_rsrp':       np.random.normal(-92, 5, n),
            'rsrp_delta':        np.random.normal(3.5, 1, n),
            'serving_rsrq':      np.random.normal(-11.5, 1, n),
            'target_rsrq':       np.random.normal(-10.5, 1, n),
            'rsrq_delta':        np.random.normal(1, 0.5, n),
            'serving_prb':       np.random.normal(45, 12, n),
            'target_prb':        np.random.normal(35, 12, n),
            'ue_speed':          np.random.normal(110, 20, n),
            'ttt':               np.random.choice([240,320], n),
            'ho_config':         np.random.choice([0,1], n),
            'ho_prep_time':      np.random.normal(130, 25, n),
            'time_in_connected': np.random.normal(7, 3, n),
        })
    elif label == 'wrong_target':
        return pd.DataFrame({
            'serving_rsrp':      np.random.normal(-90, 6, n),
            'target_rsrp':       np.random.normal(-85, 6, n),
            'rsrp_delta':        np.random.normal(5, 1.5, n),
            'serving_rsrq':      np.random.normal(-10, 1.5, n),
            'target_rsrq':       np.random.normal(-15, 1.5, n),
            'rsrq_delta':        np.random.normal(-5, 1, n),
            'serving_prb':       np.random.normal(40, 12, n),
            'target_prb':        np.random.normal(35, 10, n),
            'ue_speed':          np.random.normal(40, 20, n),
            'ttt':               np.random.choice([160,240,320], n),
            'ho_config':         np.zeros(n),
            'ho_prep_time':      np.random.normal(65, 15, n),
            'time_in_connected': np.random.normal(10, 5, n),
        })
    elif label == 'congestion_block':
        return pd.DataFrame({
            'serving_rsrp':      np.random.normal(-88, 6, n),
            'target_rsrp':       np.random.normal(-82, 6, n),
            'rsrp_delta':        np.random.normal(6, 2, n),
            'serving_rsrq':      np.random.normal(-9.5, 1.5, n),
            'target_rsrq':       np.random.normal(-8.5, 1.5, n),
            'rsrq_delta':        np.random.normal(1, 0.5, n),
            'serving_prb':       np.random.normal(45, 12, n),
            'target_prb':        np.random.normal(78, 8, n),
            'ue_speed':          np.random.normal(40, 20, n),
            'ttt':               np.random.choice([160,240,320], n),
            'ho_config':         np.random.choice([0,1], n),
            'ho_prep_time':      np.random.normal(210, 40, n),
            'time_in_connected': np.random.normal(30, 10, n),
        })
    else:  # success
        return pd.DataFrame({
            'serving_rsrp':      np.random.normal(-85, 8, n),
            'target_rsrp':       np.random.normal(-77, 8, n),
            'rsrp_delta':        np.random.normal(8, 2, n),
            'serving_rsrq':      np.random.normal(-9, 1.5, n),
            'target_rsrq':       np.random.normal(-7, 1.5, n),
            'rsrq_delta':        np.random.normal(2, 1, n),
            'serving_prb':       np.random.normal(35, 12, n),
            'target_prb':        np.random.normal(30, 10, n),
            'ue_speed':          np.random.normal(50, 30, n),
            'ttt':               np.random.choice([160,240,320], n),
            'ho_config':         np.random.choice([0,1], n),
            'ho_prep_time':      np.random.normal(50, 10, n),
            'time_in_connected': np.random.normal(45, 15, n),
        })

# Step 3 — Generate features and predict for all daily events
a3_daily_features = pd.concat([
    generate_event_features_a3(label)
    for label in a3_labels
], ignore_index=True)

# Clip to realistic bounds
a3_daily_features['serving_rsrp'] = a3_daily_features['serving_rsrp'].clip(-130, -50)
a3_daily_features['target_rsrp']  = a3_daily_features['target_rsrp'].clip(-130, -50)
a3_daily_features['serving_rsrq'] = a3_daily_features['serving_rsrq'].clip(-20, -3)
a3_daily_features['target_rsrq']  = a3_daily_features['target_rsrq'].clip(-20, -3)
a3_daily_features['target_prb']   = a3_daily_features['target_prb'].clip(0, 100)
a3_daily_features['ue_speed']     = a3_daily_features['ue_speed'].clip(0, 200)
a3_daily_features['ho_prep_time'] = a3_daily_features['ho_prep_time'].clip(10, 500)

# Predict
a3_daily_pred  = clf_a3.predict(a3_daily_features[feature_cols_a3])
a3_daily_proba = clf_a3.predict_proba(a3_daily_features[feature_cols_a3])
a3_daily_conf  = a3_daily_proba.max(axis=1)

# Step 4 — Build daily report dataframe
df_daily = pd.DataFrame({
    'event_id':    range(1, daily_failures + 1),
    'actual':      a3_labels,
    'predicted':   a3_daily_pred,
    'confidence':  a3_daily_conf,
    'priority':    [recommendations[p]['priority'] 
                    for p in a3_daily_pred],
    'action':      [recommendations[p]['action']   
                    for p in a3_daily_pred],
})

# Step 5 — Print the daily report
print(f"\nDate: 2021-07-26")
print(f"Total A3 HO failures analyzed: {daily_failures}")
print(f"Total A5 HO failures analyzed: {a5_daily_count}")
print(f"Total events processed:        {daily_failures + a5_daily_count}")

print(f"\n{'─'*60}")
print(f"A3 Root cause breakdown:")
print(f"{'─'*60}")
for cause in ['coverage_gap', 'too_late',
              'congestion_block', 'wrong_target', 'success']:
    count = (a3_daily_pred == cause).sum()
    pct   = count / daily_failures * 100
    rec   = recommendations[cause]
    bar   = '█' * int(pct / 2)
    print(f"  {cause:<20} {count:>4} ({pct:>5.1f}%)  "
          f"[{rec['priority']:<6}]  {bar}")

print(f"\n{'─'*60}")
print(f"Priority breakdown — actions required:")
print(f"{'─'*60}")
high_priority   = df_daily[df_daily['priority'] == 'high']
medium_priority = df_daily[df_daily['priority'] == 'medium']
no_action       = df_daily[df_daily['priority'] == 'none']

print(f"  High priority:   {len(high_priority):>4} failures "
      f"— dispatch field team or parameter change")
print(f"  Medium priority: {len(medium_priority):>4} failures "
      f"— schedule investigation")
print(f"  No action:       {len(no_action):>4} events  "
      f"— monitor only")

print(f"\n{'─'*60}")
print(f"Confidence distribution:")
print(f"{'─'*60}")
high_conf   = (a3_daily_conf >= 0.95).sum()
medium_conf = ((a3_daily_conf >= 0.80) & 
               (a3_daily_conf < 0.95)).sum()
low_conf    = (a3_daily_conf < 0.80).sum()
print(f"  High confidence   (>=95%): {high_conf:>4} — act immediately")
print(f"  Medium confidence (80-95%): {medium_conf:>3} — verify before action")
print(f"  Low confidence    (<80%):  {low_conf:>4} — manual review needed")

print(f"\n{'─'*60}")
print(f"Top 10 highest priority events:")
print(f"{'─'*60}")
top_events = df_daily[df_daily['priority'] == 'high'].nlargest(
    10, 'confidence')
print(f"  {'ID':<8} {'Root cause':<20} {'Confidence':<14} "
      f"{'Action'}")
print(f"  {'─'*8} {'─'*20} {'─'*14} {'─'*30}")
for _, row in top_events.head(10).iterrows():
    print(f"  {row['event_id']:<8} {row['predicted']:<20} "
          f"{row['confidence']*100:<13.1f}%  {row['action'][:40]}")

print(f"\n{'─'*60}")
print(f"Recommended actions for today:")
print(f"{'─'*60}")

action_groups = {
    'coverage_gap':     [],
    'too_late':         [],
    'congestion_block': [],
    'wrong_target':     []
}
for pred in a3_daily_pred:
    if pred in action_groups:
        action_groups[pred].append(pred)

for cause, events in action_groups.items():
    if len(events) > 0:
        rec = recommendations[cause]
        print(f"\n  [{rec['priority'].upper()}] {cause.upper()} "
              f"— {len(events)} failures")
        print(f"  Action: {rec['action']}")
        print(f"  Check:  {rec['check']}")

print(f"\n{'=' * 60}")
print(f"Report generated by: Network AI Health Monitor v1.0")
print(f"Models: A3 classifier (99.7%) + A5 classifier (99.3%)")
print(f"{'=' * 60}")

Daily HO failure analysis report
Simulating one day of network HO failures

Date: 2021-07-26
Total A3 HO failures analyzed: 500
Total A5 HO failures analyzed: 200
Total events processed:        700

────────────────────────────────────────────────────────────
A3 Root cause breakdown:
────────────────────────────────────────────────────────────
  coverage_gap          168 ( 33.6%)  [high  ]  ████████████████
  too_late              120 ( 24.0%)  [medium]  ████████████
  congestion_block      105 ( 21.0%)  [high  ]  ██████████
  wrong_target           60 ( 12.0%)  [medium]  ██████
  success                47 (  9.4%)  [none  ]  ████

────────────────────────────────────────────────────────────
Priority breakdown — actions required:
────────────────────────────────────────────────────────────
  High priority:    273 failures — dispatch field team or parameter change
  Medium priority:  180 failures — schedule investigation
  No action:         47 events  — monitor only

──────────────────